In [22]:
# import 
import requests
import json
import pandas as pd
import time
from datetime import datetime

print('imports done')

✓ Libraries imported successfully


In [29]:
# set up api
API_KEY = 'ab44dea7b323db17ba79396c3e339466'
BASE_URL = 'https://api.openweathermap.org/data/2.5/weather'

CITIES = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai','Hyderabad', 'Kolkata', 'Pune', 'Jaipur']

print(f'Got {len(CITIES)} cities to fetch')

Got 8 cities to fetch


In [30]:
# grab weather data from api
def fetch_weather(city, api_key, timeout=10):
    try:
        params = {
            'q': city,
            'appid': api_key,
            'units': 'metric'
        }
        response = requests.get(BASE_URL, params=params, timeout=timeout)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f'failed on {city}: {e}')
        return None

print('\nfetching weather-')
raw_data = []

for city in CITIES:
    data = fetch_weather(city, API_KEY)
    if data:
        raw_data.append(data)
        print(f'  {city} ok')
    time.sleep(0.5)

print(f'\ngot {len(raw_data)} cities')


fetching weather...
  Mumbai ok
  Delhi ok
  Bangalore ok
  Chennai ok
  Hyderabad ok
  Kolkata ok
  Pune ok
  Jaipur ok

got 8 cities


In [31]:
# parse the data
def parse_weather(raw_data):
    records = []
    for r in raw_data:
        try:
            record = {
                'city': r.get('name'),
                'country': r.get('sys', {}).get('country'),
                'temperature': r.get('main', {}).get('temp'),
                'feels_like': r.get('main', {}).get('feels_like'),
                'temp_min': r.get('main', {}).get('temp_min'),
                'temp_max': r.get('main', {}).get('temp_max'),
                'pressure': r.get('main', {}).get('pressure'),
                'humidity': r.get('main', {}).get('humidity'),
                'weather': r.get('weather', [{}])[0].get('main'),
                'wind_speed': r.get('wind', {}).get('speed'),
            }
            records.append(record)
        except:
            continue
    return pd.DataFrame(records)

df = parse_weather(raw_data)

# add some derived stuff
df['temperature_range'] = df['temp_max'] - df['temp_min']
df['date'] = pd.to_datetime(datetime.now()).date()

print(f'got {df.shape[0]} rows, {df.shape[1]} columns')

print(df[['city', 'temperature', 'weather']].to_string(index=False))

got 8 rows, 12 columns

preview:
     city  temperature      weather
   Mumbai        31.99         Haze
    Delhi        30.05 Thunderstorm
Bengaluru        28.86       Clouds
  Chennai        32.19       Clouds
Hyderabad        36.23       Clouds
  Kolkata        26.97       Clouds
     Pune        29.71        Clear
   Jaipur        40.62         Haze


In [26]:
# check quality
print('\nquality check:')
print(f'missing: {df.isnull().sum().sum()}')
print(f'dupes: {df.duplicated().sum()}')
print(f'types: {df.dtypes.to_dict()}')

# remove dupes
df = df.drop_duplicates()
print('cleaned')


Data Quality Checks:
Missing values: 0
Duplicates: 0
Data types:
city                  object
country               object
temperature          float64
feels_like           float64
temp_min             float64
temp_max             float64
pressure               int64
humidity               int64
weather               object
wind_speed           float64
temperature_range    float64
date                  object
dtype: object

 Data cleaned


In [27]:
# save to csv
import os

output_dir = './weather_data'
os.makedirs(output_dir, exist_ok=True)

filename = f'{output_dir}/weather_data_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
df.to_csv(filename, index=False)

print(f'\nsaved: {filename}')
print(f'size: {os.path.getsize(filename) / 1024:.2f} KB')

print('\nall data:')
print(df)


✓ Data saved to: ./weather_data/weather_data_20260528_200801.csv
File size: 0.69 KB

Final Data:
        city country  temperature  feels_like  temp_min  temp_max  pressure  \
0     Mumbai      IN        31.99       38.97     31.99     31.99      1009   
1      Delhi      IN        30.05       32.53     30.05     30.05      1000   
2  Bengaluru      IN        28.86       30.23     25.83     30.75      1012   
3    Chennai      IN        32.19       39.19     31.24     32.27      1009   
4  Hyderabad      IN        36.23       37.63     33.73     36.23      1006   
5    Kolkata      IN        26.97       28.72     26.97     26.97      1007   
6       Pune      IN        31.35       32.41     31.35     31.35      1009   
7     Jaipur      IN        40.62       40.97     40.62     40.62      1000   

   humidity       weather  wind_speed  temperature_range        date  
0        66          Haze        3.60               0.00  2026-05-28  
1        58  Thunderstorm        8.75           

In [32]:
print('\n--- STATS ---')
print(f'\ntemp (°C):')
print(f'  avg: {df["temperature"].mean():.1f}')
print(f'  low: {df["temperature"].min():.1f}')
print(f'  high: {df["temperature"].max():.1f}')

print(f'\nweather:')
print(df['weather'].value_counts())



--- STATS ---

temp (°C):
  avg: 32.1
  low: 27.0
  high: 40.6

weather:
weather
Clouds          4
Haze            2
Thunderstorm    1
Clear           1
Name: count, dtype: int64

done!
